# Import des bibliothèques nécessaires pour la connexion à la base de données et l'exécution de requêtes SQL

`import mysql.connector`

`import os`

`from pathlib import Path`

`from dotenv import load_dotenv`

`from mysql.connector import Error`

In [3]:
import mysql.connector
import pandas as pd
import os
from pathlib import Path
from mysql.connector import Error
from dotenv import load_dotenv

### On charge les dataframes contenant les phrases des romans et les motifs associés et on les envoie dans la base de données

In [13]:
# Pour la connection a la base de données:

df= pd.read_csv("../Textes + Motifs/Kock/1838_Kock-Paul-de_Madeleine_phrases_annotees.csv")

load_dotenv()

def get_db_connection():
    """Établit et retourne la connexion à la base de données MySQL."""
    try:
        conn = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            port=os.getenv("DB_PORT"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        if conn.is_connected():
            print("Connexion réussie à la base de données !")
            return conn

    except Error as e:
        print(f"Erreur lors de la connexion à MySQL : {e}")
        return None
    
    
# ID du roman auquel les phrases appartiennent, à définir en fonction de la base de données
id_roman = 74

connexion = get_db_connection()

requete = """
INSERT INTO phrase_roman (id_roman, ordre_phrase, texte_phrase, motif_phrase)
VALUES (%s, %s, %s, %s)
"""      
              
if connexion is not None:
    try:
        # On utilise la variable qui contient la connexion valide
        cursor = connexion.cursor()
        
        for _, row in df.iterrows(): # On boucle sur chaque segment de texte dans le DataFrame
            ordre_phrase = int(row["id"]) # On ajoute 1 pour que l'ordre commence à 1 au lieu de 0
            texte_phrase = row["phrase"] # Contenu du segment de texte
            texte_phrase = row["phrase"] # Contenu du segment de texte
            motif_phrase = row["motifs_phrase"] # Motif associé au segment de texte
            cursor.execute(requete,(id_roman,ordre_phrase, texte_phrase, motif_phrase)) 
        
        connexion.commit()
        
        print("Motif ajouté avec succès !")
    except Error as e:
        print(f"Erreur lors de l'exécution de la requête : {e}")
        
    finally:
        # On ferme la connexion à la base de données
        cursor.close()
        connexion.close()
        print("Connexion fermée ")
else:
    print("Script arrêté : Impossible de se connecter à la base.")

Connexion réussie à la base de données !
Motif ajouté avec succès !
Connexion fermée 
